In [1]:
import numpy as np
import plotly.graph_objects as go

# custom module
from fifth_harmony import load_data

/home/skellig/miniconda3/envs/GiQ-Hackathon26/lib/python3.12/site-packages/classiq/_internals/authentication/token_manager.py:113: UserWarning: Device is already registered.
Generating a new refresh token should only be done if the current refresh token is compromised.
To do so, set the overwrite parameter to true
  warnings.warn(


In [2]:
# Accent color per plot type — mirrors the index.html palette
ACCENT = {
    "y":    "#60c8f0",  # cyan
    "dydt": "#c8f060",  # green
    "KE":   "#f060c8",  # pink
    "PE":   "#f060c8",  # pink (energy shares accent3)
}

# Base theme colors (shared with index.html)
BG_COLOR      = "#0e0e10"
SURFACE_COLOR = "#18181c"
BORDER_COLOR  = "#2a2a30"
TEXT_COLOR    = "#e8e8e0"
MUTED_COLOR   = "#666660"

# y(t)

In [5]:
# y --- changing y0

# ==============================
# WHICH DATA TO PLOT
# ==============================

DATA_TARGET = "y"   # change to "dydt", "KE", "PE" for other plots

SLIDER = "y0"

accent = ACCENT[DATA_TARGET]

# ==============================
# SLIDER PARAMETERS + FIXED VALUES
# ==============================

vals = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
fixed   = dict(k=7, omega=1.0, vy0=1.0, b0=0.0, b1=0.5)

N0 = len(vals)
i0 = N0 // 2

# ==============================
# PRECOMPUTE TRACES
# ==============================

traces = []
for val in vals:
    try:
        time, y_expected, y_actual = load_data(**fixed, y0=val, data=DATA_TARGET, msg=False)
    except FileNotFoundError:
        time = np.array([0])
        y_expected = np.array([0])
        y_actual   = np.array([0])

    # Expected — solid accent line
    traces.append(go.Scatter(
        x=time, y=y_expected,
        visible=False,
        mode="lines",
        name="expected",
        line=dict(color=accent, width=2)
    ))

    # Actual — muted dashed line with accent markers
    traces.append(go.Scatter(
        x = time, y = y_actual,
        visible = False,
        name = "actual",
        mode = "markers",
        marker = dict(size=12),
        line = dict(color=MUTED_COLOR, dash="dash")
    ))

# Each by value now corresponds to 2 traces: [expected, actual]
# Make initial pair visible
traces[i0 * 2].visible     = True
traces[i0 * 2 + 1].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)

all_y = np.concatenate(all_y)
y_min = float(np.min(all_y))
y_max = float(np.max(all_y))
padding = (y_max - y_min) * 0.05  # 5% padding

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=f"{DATA_TARGET} — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title=DATA_TARGET,
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

fixed_text = "Fixed:<br>k = 7<br>ω = 1.0<br>vy0 = 2.0<br>b0 = 0.0<br>b1 = 0.5"

fig.update_layout(
    annotations=[dict(
        text=fixed_text,
        xref="paper", yref="paper",
        x=1.01, y=0.75,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================
# SLIDER
# ==============================

slider_0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.1,
    "len": 0.8,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": accent, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": accent,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_0])

# ==============================
# EXPORT + INJECT JS
# ==============================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N0   = {N0};
  let   idx0 = {i0};
  const plot = document.getElementById("plot");

  function update_visibility() {{
    const n_traces = N0 * 2;
    const vis      = Array(n_traces).fill(false);
    vis[idx0 * 2]     = true;   // expected
    vis[idx0 * 2 + 1] = true;   // actual
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    idx0 = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open(f"interactive_plots/interactive_{DATA_TARGET}_plot--y0.html", "w") as f:
    f.write(html)

print(f"Saved interactive_{DATA_TARGET}_plot--y0.html at interactive_plots/")

Saved interactive_y_plot--y0.html at interactive_plots/


In [6]:
# y --- changing vy0

# ==============================
# WHICH DATA TO PLOT
# ==============================

DATA_TARGET = "y"   # change to "dydt", "KE", "PE" for other plots

SLIDER = "vy0"

accent = ACCENT[DATA_TARGET]

# ==============================
# SLIDER PARAMETERS + FIXED VALUES
# ==============================

vals = np.array([1.0, 1.5, 2.0, 2.5, 3.0])
fixed   = dict(k=7, omega=1.0, y0=1.0, b0=0.0, b1=0.3)

N0 = len(vals)
i0 = N0 // 2

# ==============================
# PRECOMPUTE TRACES
# ==============================

traces = []
for val in vals:
    try:
        time, y_expected, y_actual = load_data(**fixed, vy0=val, data=DATA_TARGET, msg=False)
    except FileNotFoundError:
        time = np.array([0])
        y_expected = np.array([0])
        y_actual   = np.array([0])

    # Expected — solid accent line
    traces.append(go.Scatter(
        x=time, y=y_expected,
        visible=False,
        mode="lines",
        name="expected",
        line=dict(color=accent, width=2)
    ))

    # Actual — muted dashed line with accent markers
    traces.append(go.Scatter(
        x = time, y = y_actual,
        visible = False,
        name = "actual",
        mode = "markers",
        marker = dict(size=12),
        line = dict(color=MUTED_COLOR, dash="dash")
    ))

# Each by value now corresponds to 2 traces: [expected, actual]
# Make initial pair visible
traces[i0 * 2].visible     = True
traces[i0 * 2 + 1].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)

all_y = np.concatenate(all_y)
y_min = float(np.min(all_y))
y_max = float(np.max(all_y))
padding = (y_max - y_min) * 0.05  # 5% padding

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=f"{DATA_TARGET} — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title=DATA_TARGET,
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

fixed_text = f"Fixed:<br>k = {fixed['k']}<br>ω = {fixed['omega']}<br>y0 = {fixed['y0']}<br>b0 = {fixed['b0']}<br>b1 = {fixed['b1']}"

fig.update_layout(
    annotations=[dict(
        text=fixed_text,
        xref="paper", yref="paper",
        x=1.01, y=0.75,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================
# SLIDER
# ==============================

slider_0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.1,
    "len": 0.8,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": accent, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": accent,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_0])

# ==============================
# EXPORT + INJECT JS
# ==============================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N0   = {N0};
  let   idx0 = {i0};
  const plot = document.getElementById("plot");

  function update_visibility() {{
    const n_traces = N0 * 2;
    const vis      = Array(n_traces).fill(false);
    vis[idx0 * 2]     = true;   // expected
    vis[idx0 * 2 + 1] = true;   // actual
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    idx0 = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open(f"interactive_plots/interactive_{DATA_TARGET}_plot--{SLIDER}.html", "w") as f:
    f.write(html)

print(f"Saved interactive_{DATA_TARGET}_plot--{SLIDER}.html at interactive_plots/")

Saved interactive_y_plot--vy0.html at interactive_plots/


In [7]:
# y --- changing by

# ==============================
# WHICH DATA TO PLOT
# ==============================

DATA_TARGET = "y"   # change to "dydt", "KE", "PE" for other plots

SLIDER = "b1"

accent = ACCENT[DATA_TARGET]

# ==============================
# SLIDER PARAMETERS + FIXED VALUES
# ==============================

vals = np.round(np.arange(0.0, 1.1, 0.1), 1)
fixed = dict(k=7, omega=1.0, y0=3.0, vy0=2.0, b0=0.0)

N0 = len(vals)
i0 = N0 // 2

# ==============================
# PRECOMPUTE TRACES
# ==============================

traces = []
for val in vals:
    try:
        time, y_expected, y_actual = load_data(**fixed, b1=val, data=DATA_TARGET, msg=False)
    except FileNotFoundError:
        time = np.array([0])
        y_expected = np.array([0])
        y_actual   = np.array([0])

    # Expected — solid accent line
    traces.append(go.Scatter(
        x=time, y=y_expected,
        visible=False,
        mode="lines",
        name="expected",
        line=dict(color=accent, width=2)
    ))

    # Actual — muted dashed line with accent markers
    traces.append(go.Scatter(
        x = time, y = y_actual,
        visible = False,
        name = "actual",
        mode = "markers",
        marker = dict(size=12),
        line = dict(color=MUTED_COLOR, dash="dash")
    ))

# Each by value now corresponds to 2 traces: [expected, actual]
# Make initial pair visible
traces[i0 * 2].visible     = True
traces[i0 * 2 + 1].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)

all_y = np.concatenate(all_y)
y_min = float(np.min(all_y))
y_max = float(np.max(all_y))
padding = (y_max - y_min) * 0.05  # 5% padding

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=f"{DATA_TARGET} — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title=DATA_TARGET,
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

fixed_text = f"Fixed:<br>k = {fixed['k']}<br>ω = {fixed['omega']}<br>y0 = {fixed['y0']}<br>vy0 = {fixed['vy0']}<br>b0 = {fixed['b0']}"

fig.update_layout(
    annotations=[dict(
        text=fixed_text,
        xref="paper", yref="paper",
        x=1.01, y=0.75,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================
# SLIDER
# ==============================

slider_0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.1,
    "len": 0.8,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": accent, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": accent,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_0])

# ==============================
# EXPORT + INJECT JS
# ==============================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N0   = {N0};
  let   idx0 = {i0};
  const plot = document.getElementById("plot");

  function update_visibility() {{
    const n_traces = N0 * 2;
    const vis      = Array(n_traces).fill(false);
    vis[idx0 * 2]     = true;   // expected
    vis[idx0 * 2 + 1] = true;   // actual
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    idx0 = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open(f"interactive_plots/interactive_{DATA_TARGET}_plot--{SLIDER}.html", "w") as f:
    f.write(html)

print(f"Saved interactive_{DATA_TARGET}_plot--{SLIDER}.html at interactive_plots/")

Saved interactive_y_plot--b1.html at interactive_plots/


# y'(t)

In [8]:
# dydt --- changing y0

# ==============================
# WHICH DATA TO PLOT
# ==============================

DATA_TARGET = "dydt"   # change to "dydt", "KE", "PE" for other plots

SLIDER = "y0"

accent = ACCENT[DATA_TARGET]

# ==============================
# SLIDER PARAMETERS + FIXED VALUES
# ==============================

vals = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
fixed   = dict(k=7, omega=1.0, vy0=1.0, b0=0.0, b1=0.5)

N0 = len(vals)
i0 = N0 // 2

# ==============================
# PRECOMPUTE TRACES
# ==============================

traces = []
for val in vals:
    try:
        time, y_expected, y_actual = load_data(**fixed, y0=val, data=DATA_TARGET, msg=False)
    except FileNotFoundError:
        time = np.array([0])
        y_expected = np.array([0])
        y_actual   = np.array([0])

    # Expected — solid accent line
    traces.append(go.Scatter(
        x=time, y=y_expected,
        visible=False,
        mode="lines",
        name="expected",
        line=dict(color=accent, width=2)
    ))

    # Actual — muted dashed line with accent markers
    traces.append(go.Scatter(
        x = time, y = y_actual,
        visible = False,
        name = "actual",
        mode = "markers",
        marker = dict(size=12),
        line = dict(color=MUTED_COLOR, dash="dash")
    ))

# Each by value now corresponds to 2 traces: [expected, actual]
# Make initial pair visible
traces[i0 * 2].visible     = True
traces[i0 * 2 + 1].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)

all_y = np.concatenate(all_y)
y_min = float(np.min(all_y))
y_max = float(np.max(all_y))
padding = (y_max - y_min) * 0.05  # 5% padding

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=f"{DATA_TARGET} — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title=DATA_TARGET,
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

fixed_text = f"Fixed:<br>k = {fixed['k']}<br>ω = {fixed['omega']}<br>vy0 = {fixed['vy0']}<br>b0 = {fixed['b0']}<br>b1 = {fixed['b1']}"

fig.update_layout(
    annotations=[dict(
        text=fixed_text,
        xref="paper", yref="paper",
        x=1.01, y=0.75,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================
# SLIDER
# ==============================

slider_0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.1,
    "len": 0.8,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": accent, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": accent,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_0])

# ==============================
# EXPORT + INJECT JS
# ==============================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N0   = {N0};
  let   idx0 = {i0};
  const plot = document.getElementById("plot");

  function update_visibility() {{
    const n_traces = N0 * 2;
    const vis      = Array(n_traces).fill(false);
    vis[idx0 * 2]     = true;   // expected
    vis[idx0 * 2 + 1] = true;   // actual
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    idx0 = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open(f"interactive_plots/interactive_{DATA_TARGET}_plot--{SLIDER}.html", "w") as f:
    f.write(html)

print(f"Saved interactive_{DATA_TARGET}_plot--{SLIDER}.html at interactive_plots/")

Saved interactive_dydt_plot--y0.html at interactive_plots/


In [9]:
# dydt --- changing vy0

# ==============================
# WHICH DATA TO PLOT
# ==============================

DATA_TARGET = "dydt"   # change to "dydt", "KE", "PE" for other plots

SLIDER = "vy0"

accent = ACCENT[DATA_TARGET]

# ==============================
# SLIDER PARAMETERS + FIXED VALUES
# ==============================

vals = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
fixed   = dict(k=7, omega=1.0, y0=1.0, b0=0.0, b1=0.3)

N0 = len(vals)
i0 = N0 // 2

# ==============================
# PRECOMPUTE TRACES
# ==============================

traces = []
for val in vals:
    try:
        time, y_expected, y_actual = load_data(**fixed, vy0=val, data=DATA_TARGET, msg=False)
    except FileNotFoundError:
        time = np.array([0])
        y_expected = np.array([0])
        y_actual   = np.array([0])

    # Expected — solid accent line
    traces.append(go.Scatter(
        x=time, y=y_expected,
        visible=False,
        mode="lines",
        name="expected",
        line=dict(color=accent, width=2)
    ))

    # Actual — muted dashed line with accent markers
    traces.append(go.Scatter(
        x = time, y = y_actual,
        visible = False,
        name = "actual",
        mode = "markers",
        marker = dict(size=12),
        line = dict(color=MUTED_COLOR, dash="dash")
    ))

# Each by value now corresponds to 2 traces: [expected, actual]
# Make initial pair visible
traces[i0 * 2].visible     = True
traces[i0 * 2 + 1].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)

all_y = np.concatenate(all_y)
y_min = float(np.min(all_y))
y_max = float(np.max(all_y))
padding = (y_max - y_min) * 0.05  # 5% padding

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=f"{DATA_TARGET} — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title=DATA_TARGET,
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

fixed_text = f"Fixed:<br>k = {fixed['k']}<br>ω = {fixed['omega']}<br>y0 = {fixed['y0']}<br>b0 = {fixed['b0']}<br>b1 = {fixed['b1']}"

fig.update_layout(
    annotations=[dict(
        text=fixed_text,
        xref="paper", yref="paper",
        x=1.01, y=0.75,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================
# SLIDER
# ==============================

slider_0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.1,
    "len": 0.8,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": accent, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": accent,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_0])

# ==============================
# EXPORT + INJECT JS
# ==============================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N0   = {N0};
  let   idx0 = {i0};
  const plot = document.getElementById("plot");

  function update_visibility() {{
    const n_traces = N0 * 2;
    const vis      = Array(n_traces).fill(false);
    vis[idx0 * 2]     = true;   // expected
    vis[idx0 * 2 + 1] = true;   // actual
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    idx0 = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open(f"interactive_plots/interactive_{DATA_TARGET}_plot--{SLIDER}.html", "w") as f:
    f.write(html)

print(f"Saved interactive_{DATA_TARGET}_plot--{SLIDER}.html at interactive_plots/")

Saved interactive_dydt_plot--vy0.html at interactive_plots/


In [10]:
# dydt --- changing by

# ==============================
# WHICH DATA TO PLOT
# ==============================

DATA_TARGET = "dydt"   # change to "dydt", "KE", "PE" for other plots

SLIDER = "b1"

accent = ACCENT[DATA_TARGET]

# ==============================
# SLIDER PARAMETERS + FIXED VALUES
# ==============================

vals = np.round(np.arange(0.0, 1.1, 0.1), 1)
fixed = dict(k=7, omega=1.0, y0=3.0, vy0=2.0, b0=0.0)

N0 = len(vals)
i0 = N0 // 2

# ==============================
# PRECOMPUTE TRACES
# ==============================

traces = []
for val in vals:
    try:
        time, y_expected, y_actual = load_data(**fixed, b1=val, data=DATA_TARGET, msg=False)
    except FileNotFoundError:
        time = np.array([0])
        y_expected = np.array([0])
        y_actual   = np.array([0])

    # Expected — solid accent line
    traces.append(go.Scatter(
        x=time, y=y_expected,
        visible=False,
        mode="lines",
        name="expected",
        line=dict(color=accent, width=2)
    ))

    # Actual — muted dashed line with accent markers
    traces.append(go.Scatter(
        x = time, y = y_actual,
        visible = False,
        name = "actual",
        mode = "markers",
        marker = dict(size=12),
        line = dict(color=MUTED_COLOR, dash="dash")
    ))

# Each by value now corresponds to 2 traces: [expected, actual]
# Make initial pair visible
traces[i0 * 2].visible     = True
traces[i0 * 2 + 1].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)

all_y = np.concatenate(all_y)
y_min = float(np.min(all_y))
y_max = float(np.max(all_y))
padding = (y_max - y_min) * 0.05  # 5% padding

fig = go.Figure(traces)
fig.update_layout(
    title=dict(
        text=f"{DATA_TARGET} — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title=DATA_TARGET,
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

fixed_text = f"Fixed:<br>k = {fixed['k']}<br>ω = {fixed['omega']}<br>y0 = {fixed['y0']}<br>vy0 = {fixed['vy0']}<br>b0 = {fixed['b0']}"

fig.update_layout(
    annotations=[dict(
        text=fixed_text,
        xref="paper", yref="paper",
        x=1.01, y=0.75,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================
# SLIDER
# ==============================

slider_0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.1,
    "len": 0.8,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": accent, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": accent,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_0])

# ==============================
# EXPORT + INJECT JS
# ==============================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N0   = {N0};
  let   idx0 = {i0};
  const plot = document.getElementById("plot");

  function update_visibility() {{
    const n_traces = N0 * 2;
    const vis      = Array(n_traces).fill(false);
    vis[idx0 * 2]     = true;   // expected
    vis[idx0 * 2 + 1] = true;   // actual
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    idx0 = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open(f"interactive_plots/interactive_{DATA_TARGET}_plot--{SLIDER}.html", "w") as f:
    f.write(html)

print(f"Saved interactive_{DATA_TARGET}_plot--{SLIDER}.html at interactive_plots/")

Saved interactive_dydt_plot--b1.html at interactive_plots/


# KE and PE

In [11]:
# KE and PE --- changing y0

# Base theme colors (shared with index.html)
BG_COLOR      = "#0e0e10"
SURFACE_COLOR = "#18181c"
BORDER_COLOR  = "#2a2a30"
TEXT_COLOR    = "#e8e8e0"
MUTED_COLOR   = "#666660"

# Energy plots share accent3 (pink) from index.html, so we derive
# KE and PE as two tints of that same hue to stay coherent
KE_COLOR        = "#f060c8"   # accent3 — full pink  (KE expected)
KE_COLOR_ACTUAL = "#9e3d84"   # darker pink          (KE actual)
PE_COLOR        = "#60c8f0"   # accent2 — cyan       (PE expected, contrasts nicely)
PE_COLOR_ACTUAL = "#3d7e9e"   # darker cyan          (PE actual)

# ==============================================================================
# FIXED VALUES + SLIDER
# ==============================================================================

y0_vals = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
fixed   = dict(k=7, omega=1.0, vy0=2.0, b0=0.0, b1=0.4)
N_y0    = len(y0_vals)
i0      = N_y0 // 2

SLIDER = 'y0'

# ==============================================================================
# PRECOMPUTE TRACES
# ==============================================================================

traces = []
for y0 in y0_vals:
    try:
        time, KE_expected, KE_actual = load_data(**fixed, y0=y0, data="KE", msg=False)
        time, PE_expected, PE_actual = load_data(**fixed, y0=y0, data="PE", msg=False)
    except FileNotFoundError:
        time        = np.array([0])
        KE_expected = np.array([0])
        KE_actual   = np.array([0])
        PE_expected = np.array([0])
        PE_actual   = np.array([0])

    # KE expected — solid pink line
    traces.append(go.Scatter(
        x=time, y=KE_expected,
        visible=False,
        name="KE expected",
        mode="lines",
        line=dict(color=KE_COLOR, width=2),
    ))
    # KE actual — dark pink dashed + markers
    traces.append(go.Scatter(
        x=time, y=KE_actual,
        visible=False,
        name="KE actual",
        mode="markers",
        marker=dict(size=12, color=KE_COLOR, opacity=0.5),
        line=dict(color=KE_COLOR_ACTUAL, dash="dash", width=1.5),
    ))
    # PE expected — solid cyan line
    traces.append(go.Scatter(
        x=time, y=PE_expected,
        visible=False,
        name="PE expected",
        mode="lines",
        line=dict(color=PE_COLOR, width=2),
    ))
    # PE actual — dark cyan dashed + markers
    traces.append(go.Scatter(
        x=time, y=PE_actual,
        visible=False,
        name="PE actual",
        mode="markers",
        marker=dict(size=12, color=PE_COLOR, opacity=0.5),
        line=dict(color=PE_COLOR_ACTUAL, dash="dash", width=1.5),
    ))

# Each y0 value corresponds to 4 traces: [KE_exp, KE_act, PE_exp, PE_act]
# Make initial group visible
for offset in range(4):
    traces[i0 * 4 + offset].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)
all_y   = np.concatenate(all_y)
y_min   = float(np.min(all_y))
y_max   = float(np.max(all_y))
padding = (y_max - y_min) * 0.05         # 5% padding

fig = go.Figure(traces)

fig.update_layout(
    title=dict(
        text=f"KE and PE — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title="Energy",
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

# ==============================================================================
# FIXED PARAMETERS ANNOTATION
# ==============================================================================

fig.update_layout(
    annotations=[dict(
        text=(
            f"Fixed:<br>"
            f"k = {fixed['k']}<br>"
            f"ω = {fixed['omega']}<br>"
            f"vy0 = {fixed['vy0']}<br>"
            f"b0 = {fixed['b0']}<br>"
            f"b1 = {fixed['b1']}"
        ),
        xref="paper", yref="paper",
        x=1.01, y=0.70,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================================================================
# SLIDER
# ==============================================================================

slider_y0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.05,
    "len": 0.9,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": KE_COLOR, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": KE_COLOR,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in y0_vals
    ],
}

fig.update_layout(sliders=[slider_y0])

# ==============================================================================
# EXPORT + INJECT JS
# ==============================================================================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N_y0 = {N_y0};
  let   y0_idx = {i0};
  const plot   = document.getElementById("plot");

  function update_visibility() {{
    const vis  = Array(N_y0 * 4).fill(false);
    const base = y0_idx * 4;
    for (let offset = 0; offset < 4; offset++) {{
      vis[base + offset] = true;
    }}
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    y0_idx = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open("interactive_plots/interactive_energy_plot--y0.html", "w") as f:
    f.write(html)

print("Saved interactive_energy_plot--y0.html at interactive_plots/")

Saved interactive_energy_plot--y0.html at interactive_plots/


In [12]:
# KE and PE --- changing vy0

# Base theme colors (shared with index.html)
BG_COLOR      = "#0e0e10"
SURFACE_COLOR = "#18181c"
BORDER_COLOR  = "#2a2a30"
TEXT_COLOR    = "#e8e8e0"
MUTED_COLOR   = "#666660"

# Energy plots share accent3 (pink) from index.html, so we derive
# KE and PE as two tints of that same hue to stay coherent
KE_COLOR        = "#f060c8"   # accent3 — full pink  (KE expected)
KE_COLOR_ACTUAL = "#9e3d84"   # darker pink          (KE actual)
PE_COLOR        = "#60c8f0"   # accent2 — cyan       (PE expected, contrasts nicely)
PE_COLOR_ACTUAL = "#3d7e9e"   # darker cyan          (PE actual)

# ==============================================================================
# FIXED VALUES + SLIDER
# ==============================================================================

vals = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
fixed   = dict(k=7, omega=1.0, y0=1.0, b0=0.0, b1=0.3)
N_y0    = len(vals)
i0      = N_y0 // 2

SLIDER = 'vy0'

# ==============================================================================
# PRECOMPUTE TRACES
# ==============================================================================

traces = []
for val in vals:
    try:
        time, KE_expected, KE_actual = load_data(**fixed, vy0=val, data="KE", msg=False)
        time, PE_expected, PE_actual = load_data(**fixed, vy0=val, data="PE", msg=False)
    except FileNotFoundError:
        time        = np.array([0])
        KE_expected = np.array([0])
        KE_actual   = np.array([0])
        PE_expected = np.array([0])
        PE_actual   = np.array([0])

    # KE expected — solid pink line
    traces.append(go.Scatter(
        x=time, y=KE_expected,
        visible=False,
        name="KE expected",
        mode="lines",
        line=dict(color=KE_COLOR, width=2),
    ))
    # KE actual — dark pink dashed + markers
    traces.append(go.Scatter(
        x=time, y=KE_actual,
        visible=False,
        name="KE actual",
        mode="markers",
        marker=dict(size=12, color=KE_COLOR, opacity=0.5),
        line=dict(color=KE_COLOR_ACTUAL, dash="dash", width=1.5),
    ))
    # PE expected — solid cyan line
    traces.append(go.Scatter(
        x=time, y=PE_expected,
        visible=False,
        name="PE expected",
        mode="lines",
        line=dict(color=PE_COLOR, width=2),
    ))
    # PE actual — dark cyan dashed + markers
    traces.append(go.Scatter(
        x=time, y=PE_actual,
        visible=False,
        name="PE actual",
        mode="markers",
        marker=dict(size=12, color=PE_COLOR, opacity=0.5),
        line=dict(color=PE_COLOR_ACTUAL, dash="dash", width=1.5),
    ))

# Each y0 value corresponds to 4 traces: [KE_exp, KE_act, PE_exp, PE_act]
# Make initial group visible
for offset in range(4):
    traces[i0 * 4 + offset].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)
all_y   = np.concatenate(all_y)
y_min   = float(np.min(all_y))
y_max   = float(np.max(all_y))
padding = (y_max - y_min) * 0.05         # 5% padding

fig = go.Figure(traces)

fig.update_layout(
    title=dict(
        text=f"KE and PE — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title="Energy",
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

# ==============================================================================
# FIXED PARAMETERS ANNOTATION
# ==============================================================================

fig.update_layout(
    annotations=[dict(
        text=(
            f"Fixed:<br>"
            f"k = {fixed['k']}<br>"
            f"ω = {fixed['omega']}<br>"
            f"y0 = {fixed['y0']}<br>"
            f"b0 = {fixed['b0']}<br>"
            f"b1 = {fixed['b1']}"
        ),
        xref="paper", yref="paper",
        x=1.01, y=0.70,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================================================================
# SLIDER
# ==============================================================================

slider_y0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.05,
    "len": 0.9,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": KE_COLOR, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": KE_COLOR,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_y0])

# ==============================================================================
# EXPORT + INJECT JS
# ==============================================================================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N_y0 = {N_y0};
  let   y0_idx = {i0};
  const plot   = document.getElementById("plot");

  function update_visibility() {{
    const vis  = Array(N_y0 * 4).fill(false);
    const base = y0_idx * 4;
    for (let offset = 0; offset < 4; offset++) {{
      vis[base + offset] = true;
    }}
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    y0_idx = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open("interactive_plots/interactive_energy_plot--vy0.html", "w") as f:
    f.write(html)

print("Saved interactive_energy_plot--vy0.html at interactive_plots/")

Saved interactive_energy_plot--vy0.html at interactive_plots/


In [13]:
# KE and PE --- changing by

# Base theme colors (shared with index.html)
BG_COLOR      = "#0e0e10"
SURFACE_COLOR = "#18181c"
BORDER_COLOR  = "#2a2a30"
TEXT_COLOR    = "#e8e8e0"
MUTED_COLOR   = "#666660"

# Energy plots share accent3 (pink) from index.html, so we derive
# KE and PE as two tints of that same hue to stay coherent
KE_COLOR        = "#f060c8"   # accent3 — full pink  (KE expected)
KE_COLOR_ACTUAL = "#9e3d84"   # darker pink          (KE actual)
PE_COLOR        = "#60c8f0"   # accent2 — cyan       (PE expected, contrasts nicely)
PE_COLOR_ACTUAL = "#3d7e9e"   # darker cyan          (PE actual)

# ==============================================================================
# FIXED VALUES + SLIDER
# ==============================================================================

vals = np.round(np.arange(0.0, 1.1, 0.1), 1)
fixed = dict(k=7, omega=1.0, y0=3.0, vy0=2.0, b0=0.0)
N_y0    = len(vals)
i0      = N_y0 // 2

SLIDER = 'b1'

# ==============================================================================
# PRECOMPUTE TRACES
# ==============================================================================

traces = []
for val in vals:
    try:
        time, KE_expected, KE_actual = load_data(**fixed, b1=val, data="KE", msg=False)
        time, PE_expected, PE_actual = load_data(**fixed, b1=val, data="PE", msg=False)
    except FileNotFoundError:
        time        = np.array([0])
        KE_expected = np.array([0])
        KE_actual   = np.array([0])
        PE_expected = np.array([0])
        PE_actual   = np.array([0])

    # KE expected — solid pink line
    traces.append(go.Scatter(
        x=time, y=KE_expected,
        visible=False,
        name="KE expected",
        mode="lines",
        line=dict(color=KE_COLOR, width=2),
    ))
    # KE actual — dark pink markers
    traces.append(go.Scatter(
        x=time, y=KE_actual,
        visible=False,
        name="KE actual",
        mode="markers",
        marker=dict(size=12, color=KE_COLOR, opacity=0.5),
        line=dict(color=KE_COLOR_ACTUAL, dash="dash", width=1.5),
    ))
    # PE expected — solid cyan line
    traces.append(go.Scatter(
        x=time, y=PE_expected,
        visible=False,
        name="PE expected",
        mode="lines",
        line=dict(color=PE_COLOR, width=2),
    ))
    # PE actual — dark cyan markers
    traces.append(go.Scatter(
        x=time, y=PE_actual,
        visible=False,
        name="PE actual",
        mode="markers",
        marker=dict(size=12, color=PE_COLOR, opacity=0.5),
        line=dict(color=PE_COLOR_ACTUAL, dash="dash", width=1.5),
    ))

# Each y0 value corresponds to 4 traces: [KE_exp, KE_act, PE_exp, PE_act]
# Make initial group visible
for offset in range(4):
    traces[i0 * 4 + offset].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)
all_y   = np.concatenate(all_y)
y_min   = float(np.min(all_y))
y_max   = float(np.max(all_y))
padding = (y_max - y_min) * 0.05         # 5% padding

fig = go.Figure(traces)

fig.update_layout(
    title=dict(
        text=f"KE and PE — expected vs actual<br><sup>changing {SLIDER}</sup>",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title="Energy",
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

# ==============================================================================
# FIXED PARAMETERS ANNOTATION
# ==============================================================================

fig.update_layout(
    annotations=[dict(
        text=(
            f"Fixed:<br>"
            f"k = {fixed['k']}<br>"
            f"ω = {fixed['omega']}<br>"
            f"y0 = {fixed['y0']}<br>"
            f"vy0 = {fixed['vy0']}<br>"
            f"b0 = {fixed['b0']}"
        ),
        xref="paper", yref="paper",
        x=1.01, y=0.70,
        xanchor="left", yanchor="top",
        showarrow=False,
        font=dict(size=13, color=MUTED_COLOR, family="monospace"),
        align="left",
    )]
)

# ==============================================================================
# SLIDER
# ==============================================================================

slider_y0 = {
    "active": i0,
    "y": -0.15,
    "x": 0.05,
    "len": 0.9,
    "currentvalue": {
        "prefix": f"{SLIDER} = ",
        "font": {"color": KE_COLOR, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": KE_COLOR,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vals
    ],
}

fig.update_layout(sliders=[slider_y0])

# ==============================================================================
# EXPORT + INJECT JS
# ==============================================================================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
  const N_y0 = {N_y0};
  let   y0_idx = {i0};
  const plot   = document.getElementById("plot");

  function update_visibility() {{
    const vis  = Array(N_y0 * 4).fill(false);
    const base = y0_idx * 4;
    for (let offset = 0; offset < 4; offset++) {{
      vis[base + offset] = true;
    }}
    Plotly.restyle(plot, {{visible: vis}});
  }}

  plot.on('plotly_sliderchange', function(e) {{
    y0_idx = e.step._index;
    update_visibility();
  }});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open(f"interactive_plots/interactive_energy_plot--{SLIDER}.html", "w") as f:
    f.write(html)

print(f"Saved interactive_energy_plot--{SLIDER}.html at interactive_plots/")

Saved interactive_energy_plot--b1.html at interactive_plots/


In [17]:
# KE and PE --- changing y0, vy0 and by

# Base theme colors (shared with index.html)
BG_COLOR      = "#0e0e10"
SURFACE_COLOR = "#18181c"
BORDER_COLOR  = "#2a2a30"
TEXT_COLOR    = "#e8e8e0"
MUTED_COLOR   = "#666660"

# Energy plots share accent3 (pink) from index.html, so we derive
# KE and PE as two tints of that same hue to stay coherent
KE_COLOR        = "#f060c8"   # accent3 — full pink  (KE expected)
KE_COLOR_ACTUAL = "#9e3d84"   # darker pink          (KE actual)
PE_COLOR        = "#60c8f0"   # accent2 — cyan       (PE expected, contrasts nicely)
PE_COLOR_ACTUAL = "#3d7e9e"   # darker cyan          (PE actual)


# ==============================
# FIXED VALUES + SLIDER
# ==============================

y0_vals = np.array([1.0, 1.5, 2.0, 3.0])
vy0_vals = np.array([1.0, 1.5, 2.0, 3.0])
by_vals  = np.round(np.arange(0.0, 1.1, 0.1), 1)
fixed    = dict(k=7, omega=1.0, b0=0.0)

N_y0 = len(y0_vals)
N_vy0 = len(vy0_vals)
N_b1 = len(by_vals)

i0 = N_y0 // 2
j0 = N_vy0 // 2
k0 = N_b1 // 2

# ==============================
# PRECOMPUTE TRACES
# ==============================

traces = []
for y0 in y0_vals:
    for vy0 in vy0_vals:
        for by in by_vals:
            try:
                time, KE_expected, KE_actual = load_data(**fixed, y0=y0, vy0=vy0, b1=by, data="KE", msg=False)
                time, PE_expected, PE_actual = load_data(**fixed, y0=y0, vy0=vy0, b1=by, data="PE", msg=False)
            except FileNotFoundError:
                time        = np.array([0])
                KE_expected = np.array([0])
                KE_actual   = np.array([0])
                PE_expected = np.array([0])
                PE_actual   = np.array([0])
        
            # 4 traces per by value, always in this order: KE_exp, KE_act, PE_exp, PE_act
            # KE expected — solid pink line
            traces.append(go.Scatter(
                x=time, y=KE_expected,
                visible=False,
                name="KE expected",
                mode="lines",
                line=dict(color=KE_COLOR, width=2),
            ))
            # KE actual — dark pink dashed + markers
            traces.append(go.Scatter(
                x=time, y=KE_actual,
                visible=False,
                name="KE actual",
                mode="markers",
                marker=dict(size=12, color=KE_COLOR, opacity=0.5),
                line=dict(color=KE_COLOR_ACTUAL, dash="dash", width=1.5),
            ))
            # PE expected — solid cyan line
            traces.append(go.Scatter(
                x=time, y=PE_expected,
                visible=False,
                name="PE expected",
                mode="lines",
                line=dict(color=PE_COLOR, width=2),
            ))
            # PE actual — dark cyan dashed + markers
            traces.append(go.Scatter(
                x=time, y=PE_actual,
                visible=False,
                name="PE actual",
                mode="markers",
                marker=dict(size=12, color=PE_COLOR, opacity=0.5),
                line=dict(color=PE_COLOR_ACTUAL, dash="dash", width=1.5),
            ))

# Make initial group of 4 visible
base = (i0 * N_vy0 * N_b1 + j0 * N_b1 + k0) * 4
for offset in range(4):
    traces[base + offset].visible = True

all_y = []
for trace in traces:
    y = np.array(trace.y)
    if not (len(y) == 1 and y[0] == 0):  # skip placeholders
        all_y.append(y)
all_y   = np.concatenate(all_y)
y_min   = float(np.min(all_y))
y_max   = float(np.max(all_y))
padding = (y_max - y_min) * 0.05         # 5% padding

fig = go.Figure(traces)

fig.update_layout(
    title=dict(
        text="KE and PE — expected vs actual",
        font=dict(family="serif", size=20, color=TEXT_COLOR),
        x=0.05,
    ),
    xaxis_title="time",
    yaxis_title="Energy",
    yaxis=dict(
        range=[y_min - padding, y_max + padding],
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    xaxis=dict(
        gridcolor=BORDER_COLOR,
        zerolinecolor=BORDER_COLOR,
        tickfont=dict(color=MUTED_COLOR),
        title_font=dict(color=MUTED_COLOR),
    ),
    plot_bgcolor=SURFACE_COLOR,
    paper_bgcolor=BG_COLOR,
    font=dict(family="monospace", color=TEXT_COLOR),
    legend=dict(
        bgcolor=SURFACE_COLOR,
        bordercolor=BORDER_COLOR,
        borderwidth=1,
        font=dict(color=TEXT_COLOR),
    ),
    margin=dict(r=180),
)

# ==============================
# SLIDER
# ==============================

slider_y0 = {
    "active": i0,
    "y": -0.05,
    "x": 0.05,
    "len": 0.4,
    "currentvalue": {
        "prefix": "y0 = ",
        "font": {"color": KE_COLOR, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": KE_COLOR,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in y0_vals
    ],
}

slider_vy = {
    "active": j0,
    "y": -0.05,
    "x": 0.55,
    "len": 0.4,
    "currentvalue": {
        "prefix": "vy0 = ",
        "font": {"color": KE_COLOR, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": KE_COLOR,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in vy0_vals
    ],
}

slider_b1 = {
    "active": k0,
    "y": -0.30,
    "x": 0.05,
    "len": 0.9,
    "currentvalue": {
        "prefix": "b1 = ",
        "font": {"color": KE_COLOR, "size": 14, "family": "monospace"},
    },
    "font": {"color": MUTED_COLOR, "family": "monospace"},
    "bgcolor": SURFACE_COLOR,
    "bordercolor": BORDER_COLOR,
    "activebgcolor": KE_COLOR,
    "steps": [
        {"label": f"{v:.1f}", "method": "skip"}
        for v in by_vals
    ],
}

fig.update_layout(sliders=[slider_y0, slider_vy, slider_b1])

# ==============================
# EXPORT + INJECT JS
# ==============================

html = fig.to_html(include_plotlyjs="cdn", full_html=True, div_id="plot")

js = f"""
<script>
const N_y0 = {N_y0};
const N_vy0 = {N_vy0};
const N_b1 = {N_b1};

let y0_idx = {i0};
let vy_idx = {j0};
let by_idx = {k0};

const plot = document.getElementById("plot");

function update_visibility() {{
    const n_traces = N_y0 * N_vy0 * N_b1 * 4;
    const vis = Array(n_traces).fill(false);
    const base = (y0_idx * N_vy0 * N_b1 + vy_idx * N_b1 + by_idx) * 4;
    for (let offset = 0; offset < 4; offset++) {{
        vis[base + offset] = true;
    }}
    Plotly.restyle(plot, {{visible: vis}});
}}

plot.on('plotly_sliderchange', function(e) {{
    const sliderIndex = e.slider._index;
    const stepIndex   = e.step._index;
    if      (sliderIndex === 0) {{ y0_idx = stepIndex; }}
    else if (sliderIndex === 1) {{ vy_idx = stepIndex; }}
    else if (sliderIndex === 2) {{ by_idx = stepIndex; }}
    update_visibility();
}});
</script>
"""

html = html.replace("</body>", js + "\n</body>")

with open("interactive_plots/interactive_energy_plot.html", "w") as f:
    f.write(html)

print("Saved interactive_energy_plot.html at interactive_plots/")

Saved interactive_energy_plot.html at interactive_plots/
